This is an end-to-end Tensorflow preprocessing pipeline for PDF documenents

In [29]:
# Imports and NLTK setup
import numpy
import tensorflow as tf
import nltk
import re

from PyPDF2 import PdfReader
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Downloading required NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

STOPWORDS = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\E1012131\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\E1012131\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\E1012131\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


In [30]:
# PDF Text Extraction
def extract_pages_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    pages = []

    for page in reader.pages:
        pages.append(page.extract_text())
    return pages

In [31]:
# PDF Text Cleaning
def clean_page_text(text):
    text = text.lower() # converts text to lowercase
    text = re.sub(r'\bpage\s+\d+\b', '', text) # removes page numbers
    text = re.sub(r'\b\d+\b', '', text) # removes other numbers
    text = re.sub(r'[^a-z\s]', '', text) # removes special characters
    tokens = word_tokenize(text) # tokenize
    cleaned_tokens = [token for token in tokens if token not in STOPWORDS and len(token) > 2] # remove stopwords and short tokens
    
    return cleaned_tokens

In [32]:
# Tensorflow tf.data pipeline
def tf_clean_text(text):
    tokens = tf.py_function(
        func = lambda x: clean_page_text(x.numpy().decode('utf-8')),
        inp = [text],
        Tout = tf.string
    )
    return tokens

In [33]:
# Load PDF 
PDF_PATH = "C:/Users/E1012131/Downloads/Introduction to NLP Course (1).pdf"
pages = extract_pages_from_pdf(PDF_PATH)
print(f"Number of pages extracted: {len(pages)}")


Number of pages extracted: 24


In [34]:
# Build tf.data Dataset
dataset = tf.data.Dataset.from_tensor_slices(pages) # Create tensorfloe dataset
dataset = dataset.map(
    lambda x: tf_clean_text(x),
    num_parallel_calls=tf.data.AUTOTUNE
) # parallel processing
dataset = dataset.prefetch(tf.data.AUTOTUNE) # performance optimization

In [ ]:
# inspect cleaned output tokenes
for i, tokens in enumerate(dataset.take(10)):
    print(f"\n--- Page {i + 1} Tokens ---")
    print(tokens.numpy())


--- Page 1 Tokens ---
b'advanced'

--- Page 2 Tokens ---
b'fridays'

--- Page 3 Tokens ---
b'course'

--- Page 4 Tokens ---
b'nlp'

--- Page 5 Tokens ---
b'nlp'

--- Page 6 Tokens ---
b'motivation'

--- Page 7 Tokens ---
b'motivation'

--- Page 8 Tokens ---
b'motivation'

--- Page 9 Tokens ---
b'motivation'

--- Page 10 Tokens ---
b'requirements'
